In [2]:
import requests
import json

#define the countries and indicators you want to retrieve data for
countries = {
	'KE': 'Kenya',
	'RW': 'Rwanda',
	'TZ': 'Tanzania'
}

indicators = {
	'NY.GDP.MKTP.CD': 'GDP',
	'SP.POP.TOTL': 'Population',
	'FP.CPI.TOTL.ZG': 'Inflation'
}

all_data = {}

#loop through each country and indicator to retrieve data from the API
for code, name in countries.items():
	all_data[name] = {}

	for indicator_code, indicator_name in indicators.items():
		url = f'https://api.worldbank.org/v2/country/{code}/indicator/{indicator_code}?date=2000:2022&format=json&per_page=1000'
		response = requests.get(url)

		if response.status_code == 200:
			data = response.json()

			#The World Bank API returns metadata in [0] and the actual data in [1]
			if len(data) > 1:
				all_data[name][indicator_name] = data[1]
				print(f"✅ Fetched {len(data[1])} rows for {name}")
		else:
			print(f"❌ Failed to fetch {name}. Status: {response.status_code}")

print(f"\nTotal records fetched: {len(all_data)}")
				

✅ Fetched 23 rows for Kenya
✅ Fetched 23 rows for Kenya
✅ Fetched 23 rows for Kenya
✅ Fetched 23 rows for Rwanda
✅ Fetched 23 rows for Rwanda
✅ Fetched 23 rows for Rwanda
✅ Fetched 23 rows for Tanzania
✅ Fetched 23 rows for Tanzania
✅ Fetched 23 rows for Tanzania

Total records fetched: 3


In [5]:
import pandas as pd

# Parse the nested JSON into a flat list of dictionaries
parsed_data = []
for country_name, country_data in all_data.items():
    for indicator_name, rows in country_data.items():
        for row in rows:
            parsed_data.append({
                'country_code': row['countryiso3code'],
                'country_name': country_name,
                'year': int(row['date']),
                'indicator_code': row['indicator']['id'],
                'indicator_name': indicator_name,
                'value': row['value']  # This can be None/NaN
            })

df = pd.DataFrame(parsed_data)

# Inspect the data
print("Raw DataFrame shape:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())

Raw DataFrame shape: (207, 6)
  country_code country_name  year  indicator_code indicator_name         value
0          KEN        Kenya  2022  NY.GDP.MKTP.CD            GDP  1.144490e+11
1          KEN        Kenya  2021  NY.GDP.MKTP.CD            GDP  1.097037e+11
2          KEN        Kenya  2020  NY.GDP.MKTP.CD            GDP  1.006575e+11
3          KEN        Kenya  2019  NY.GDP.MKTP.CD            GDP  1.003784e+11
4          KEN        Kenya  2018  NY.GDP.MKTP.CD            GDP  9.220298e+10

Missing values:
 country_code      0
country_name      0
year              0
indicator_code    0
indicator_name    0
value             0
dtype: int64


In [6]:
# Drop rows where the value is missing (NaN)
df = df.dropna(subset=['value'])

# Drop duplicates (just in case the API returned overlapping data)
df = df.drop_duplicates(subset=['country_code', 'year', 'indicator_code'])

print("Cleaned DataFrame shape:", df.shape)
print("Missing values after cleaning:", df.isnull().sum().sum())

Cleaned DataFrame shape: (207, 6)
Missing values after cleaning: 0


In [7]:
import sqlite3

# Connect to a new SQLite database (this creates the file)
conn = sqlite3.connect('economic_data.db')
cursor = conn.cursor()

# 1. Create Tables
cursor.execute('''
CREATE TABLE IF NOT EXISTS Countries (
    country_code TEXT PRIMARY KEY,
    country_name TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Indicators (
    indicator_code TEXT PRIMARY KEY,
    indicator_name TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS EconomicData (
    country_code TEXT,
    indicator_code TEXT,
    year INTEGER,
    value REAL,
    FOREIGN KEY (country_code) REFERENCES Countries(country_code),
    FOREIGN KEY (indicator_code) REFERENCES Indicators(indicator_code),
    PRIMARY KEY (country_code, indicator_code, year)
)
''')

# 2. Populate Countries table
countries_df = df[['country_code', 'country_name']].drop_duplicates()
cursor.executemany('INSERT OR IGNORE INTO Countries VALUES (?, ?)', countries_df.values.tolist())

# 3. Populate Indicators table
indicators_df = df[['indicator_code', 'indicator_name']].drop_duplicates()
cursor.executemany('INSERT OR IGNORE INTO Indicators VALUES (?, ?)', indicators_df.values.tolist())

# 4. Populate EconomicData table
economic_df = df[['country_code', 'indicator_code', 'year', 'value']]
cursor.executemany('INSERT OR IGNORE INTO EconomicData VALUES (?, ?, ?, ?)', economic_df.values.tolist())

conn.commit()
print("✅ Database created and populated successfully!")

✅ Database created and populated successfully!


query1 = """
SELECT c.country_name, ROUND(AVG(e.value), 2) as avg_gdp
FROM EconomicData e
JOIN Countries c ON e.country_code = c.country_code
WHERE e.indicator_code = 'NY.GDP.PCAP.CD'
  AND e.year BETWEEN 2010 AND 2020
GROUP BY c.country_name
ORDER BY avg_gdp DESC
"""
print("1. Average GDP per Capita (2010-2020):")
print(pd.read_sql_query(query1, conn), "\n")

query2 = """
SELECT c.country_name, e.value as co2_emissions
FROM EconomicData e
JOIN Countries c ON e.country_code = c.country_code
WHERE e.indicator_code = 'EN.ATM.CO2E.PC'
  AND e.year = 2020
ORDER BY co2_emissions DESC
"""
print("2. CO2 Emissions per Capita in 2020:")
print(pd.read_sql_query(query2, conn), "\n")

query3 = """
SELECT c.country_name,
       MAX(CASE WHEN e.year = 2000 THEN e.value END) AS pop_2000,
       MAX(CASE WHEN e.year = 2022 THEN e.value END) AS pop_2022,
       ROUND(((MAX(CASE WHEN e.year = 2022 THEN e.value END) - MAX(CASE WHEN e.year = 2000 THEN e.value END)) / MAX(CASE WHEN e.year = 2000 THEN e.value END)) * 100, 2) AS growth_pct
FROM EconomicData e
JOIN Countries c ON e.country_code = c.country_code
WHERE e.indicator_code = 'SP.POP.TOTL'
GROUP BY c.country_name
"""
print("3. Population Growth (2000 vs 2022):")
print(pd.read_sql_query(query3, conn), "\n")

query4 = """
SELECT c.country_name, e.year, e.value as gdp
FROM EconomicData e
JOIN Countries c ON e.country_code = c.country_code
WHERE e.indicator_code = 'NY.GDP.PCAP.CD'
  AND e.value = (
      SELECT MAX(e2.value)
      FROM EconomicData e2
      WHERE e2.country_code = e.country_code
        AND e2.indicator_code = 'NY.GDP.PCAP.CD'
  )
ORDER BY gdp DESC
"""
print("4. Peak GDP per Capita Year for Each Country:")
print(pd.read_sql_query(query4, conn), "\n")

In [8]:
# Save the cleaned API data to CSV as well
df.to_csv('world_bank_cleaned_data.csv', index=False)
conn.close()
print("✅ Notebook complete. Files saved.")

✅ Notebook complete. Files saved.
